In [ ]:
webhook_url = ""  # Optional Discord/Teams webhook URL for alerts
fail_pipeline_on_error = True

In [3]:
import json
import urllib.request
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, IntegerType

webhook_url = globals().get("webhook_url", "")
fail_pipeline_on_error = globals().get("fail_pipeline_on_error", True)

run_timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
validation_records = []


class SilverQualitySuite:
    def __init__(self, table_name: str):
        self.table_name = table_name
        self.df = spark.table(table_name).cache()
        self.total_rows = self.df.count()
        self.results = []

    def _record(self, rule: str, passed: bool, detail: str):
        self.results.append({"rule": rule, "passed": bool(passed), "detail": detail})

    def expect_row_count_between(self, min_val: int, max_val: int):
        passed = min_val <= self.total_rows <= max_val
        self._record("row_count_between", passed, f"count={self.total_rows}, expected=[{min_val}, {max_val}]")

    def expect_not_null(self, col_name: str, mostly: float = 1.0):
        valid = self.df.filter(F.col(col_name).isNotNull()).count()
        ratio = (valid / self.total_rows) if self.total_rows > 0 else 0.0
        self._record(f"not_null:{col_name}", ratio >= mostly, f"ratio={ratio:.5f}, threshold={mostly}")

    def expect_between(self, col_name: str, min_val: float, max_val: float, mostly: float = 1.0):
        valid = self.df.filter(
            F.col(col_name).isNotNull() & (F.col(col_name) >= min_val) & (F.col(col_name) <= max_val)
        ).count()
        ratio = (valid / self.total_rows) if self.total_rows > 0 else 0.0
        self._record(f"between:{col_name}", ratio >= mostly, f"ratio={ratio:.5f}, range=[{min_val}, {max_val}]")

    def expect_in_set(self, col_name: str, allowed: list, mostly: float = 1.0):
        valid = self.df.filter(F.col(col_name).isin(allowed)).count()
        ratio = (valid / self.total_rows) if self.total_rows > 0 else 0.0
        self._record(f"in_set:{col_name}", ratio >= mostly, f"ratio={ratio:.5f}, allowed={allowed}")

    def expect_unique(self, col_name: str):
        distinct_cnt = self.df.select(col_name).distinct().count()
        passed = (self.total_rows > 0) and (distinct_cnt == self.total_rows)
        self._record(f"unique:{col_name}", passed, f"distinct={distinct_cnt}, total={self.total_rows}")

    def finalize(self) -> bool:
        self.df.unpersist()
        evaluated = len(self.results)
        succeeded = sum(1 for r in self.results if r["passed"])
        failed = evaluated - succeeded
        overall_passed = (failed == 0)

        validation_records.append((
            run_timestamp,
            self.table_name,
            overall_passed,
            evaluated,
            succeeded,
            failed,
            json.dumps(self.results)[:4000]
        ))
        print(f"[{self.table_name}] Passed: {overall_passed} ({succeeded}/{evaluated} expectations)")
        return overall_passed


# 1. NYC Taxi Silver
s_taxi = SilverQualitySuite("nyc_taxi_silver")
s_taxi.expect_row_count_between(1000000, 50000000)
s_taxi.expect_not_null("pickup_date")
s_taxi.expect_between("trip_distance", 0, 500, mostly=0.999)
s_taxi.expect_between("total_amount", 0.01, 5000, mostly=0.999)
s_taxi.expect_between("pickup_hour", 0, 23)
taxi_ok = s_taxi.finalize()

# 2. OpenAQ Air Quality Silver
s_aq = SilverQualitySuite("openaq_measurements_silver")
s_aq.expect_row_count_between(100, 100000)
s_aq.expect_not_null("date_key")
s_aq.expect_in_set("parameter_name", ["pm25", "no2"])
s_aq.expect_between("daily_avg_value", 0, 1000)
aq_ok = s_aq.finalize()

# 3. ECB FX Rates Silver
s_fx = SilverQualitySuite("fx_usd_eur_silver")
s_fx.expect_row_count_between(1000, 50000)
s_fx.expect_not_null("rate_date")
s_fx.expect_between("exchange_rate", 0.5, 2.5)
fx_ok = s_fx.finalize()

# 4. World Bank GDP Silver
s_gdp = SilverQualitySuite("gdp_silver")
s_gdp.expect_row_count_between(30, 500)
s_gdp.expect_in_set("country_iso3", ["USA"])
s_gdp.expect_between("gdp_usd", 1e11, 1e14)
gdp_ok = s_gdp.finalize()

# 5. NYC Taxi Zones Silver
s_zones = SilverQualitySuite("nyc_taxi_zones_silver")
s_zones.expect_row_count_between(260, 270)
s_zones.expect_unique("location_id")
s_zones.expect_not_null("borough", mostly=0.99)
zones_ok = s_zones.finalize()

# Persist validation audit log to Delta table in Lakehouse_Silver
log_schema = StructType([
    StructField("run_timestamp_utc", StringType(), False),
    StructField("table_name", StringType(), False),
    StructField("validation_passed", BooleanType(), False),
    StructField("evaluated_expectations", IntegerType(), False),
    StructField("successful_expectations", IntegerType(), False),
    StructField("failed_expectations", IntegerType(), False),
    StructField("details_json", StringType(), True)
])

df_logs = spark.createDataFrame(validation_records, schema=log_schema)
df_logs.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("data_quality_logs")
display(df_logs.drop("details_json"))

all_passed = all([taxi_ok, aq_ok, fx_ok, gdp_ok, zones_ok])

if webhook_url:
    status_label = "PASSED" if all_passed else "FAILED"
    req = urllib.request.Request(
        webhook_url,
        data=json.dumps({"content": f"Microsoft Fabric DQ Gate [{status_label}] at {run_timestamp}"}).encode("utf-8"),
        headers={"Content-Type": "application/json"}
    )
    urllib.request.urlopen(req, timeout=15)

if not all_passed and fail_pipeline_on_error:
    raise RuntimeError("Data quality validation failed on one or more Silver tables. Halting Gold warehouse load.")

print("All Silver data quality checks passed successfully.")

StatementMeta(, e27806aa-c233-4fa6-b5f4-4768e7bc9c07, 10, Finished, Available, Finished, False)

[nyc_taxi_silver] Passed: True (5/5 expectations)
[openaq_measurements_silver] Passed: True (4/4 expectations)
[fx_usd_eur_silver] Passed: True (3/3 expectations)
[gdp_silver] Passed: True (3/3 expectations)
[nyc_taxi_zones_silver] Passed: True (3/3 expectations)


SynapseWidget(Synapse.DataFrame, fb155646-150b-4825-bf44-f9eb173b7efd)

All Great Expectations data quality checks passed successfully.
